In [13]:
import pandas as pd
from pathlib import Path
from io import StringIO
import re

# --- Expected columns ---
COLUMNS = ["Date(UTC)", "User Name", "User Email", "View Duration (minutes)"]

# --- Files to merge ---
files = [
    "zoomus_recording_analytics_10-27-2025-part1.csv",
    "zoomus_recording_analytics_10-27-2025-part2.csv",
]

# --- Read + clean helper ---
def read_and_clean(path):
    # Read raw text and remove a single trailing comma at end of each non-empty line
    txt = Path(path).read_text(encoding='utf-8', errors='replace')
    # remove a single trailing comma at the end of each line (preserve internal commas inside quotes)
    txt = re.sub(r",\s*$", "", txt, flags=re.MULTILINE)
    # Now parse via pandas from the cleaned text
    df = pd.read_csv(StringIO(txt), dtype=str)
    # clean headers: remove BOM/RTL and strip
    df.columns = (
        df.columns.astype(str)
        .str.replace('\ufeff', '', regex=True)
        .str.replace('\u200e', '', regex=True)
        .str.strip()
    )
    # If the file had extra empty trailing column, drop unnamed empty columns
    unnamed = [c for c in df.columns if c.startswith('Unnamed')]
    if unnamed:
        df = df.drop(columns=unnamed)
    # Ensure we have the expected columns (if there are extras, keep only expected)
    present = [c for c in COLUMNS if c in df.columns]
    missing = [c for c in COLUMNS if c not in df.columns]
    if missing:
        raise KeyError(f"Missing expected columns {missing} in {path}. Found: {df.columns.tolist()}")
    return df[present]

# --- Load ---
dfs = []
for f in files:
    p = Path(f)
    if not p.exists():
        raise FileNotFoundError(f"Missing input file: {f}")
    dfs.append(read_and_clean(p))

combined = pd.concat(dfs, ignore_index=True)

# --- Diagnostics (quick) ---
print("Columns:", combined.columns.tolist())
print("Shape:", combined.shape)
print(combined.head(10).to_string(index=False))

# --- Normalize duration to numeric (handle "< 1") ---
dur_col = "View Duration (minutes)"
combined[dur_col] = (
    combined[dur_col].astype(str).str.strip().replace({"< 1": "0.5", "<1": "0.5"})
)
combined[dur_col] = pd.to_numeric(combined[dur_col], errors="coerce").fillna(0)

# --- Define keys ---
KEYS = ["User Name", "User Email"]

# --- Aggregate durations ---
sum_dur = (
    combined
    .groupby(KEYS, as_index=False, sort=False)[dur_col]
    .sum()
)

# --- Aggregate dates into a single TEXT field ---
dates_text = (
    combined
    .assign(**{"Date(UTC)": combined["Date(UTC)"].astype(str).str.strip()})
    .groupby(KEYS, as_index=False, sort=False)["Date(UTC)"]
    .agg(lambda s: ", ".join(sorted(pd.unique([x for x in s.dropna() if x]))))
    .rename(columns={"Date(UTC)": "All Dates (UTC)"})
)

# --- Final merge ---
final = sum_dur.merge(dates_text, on=KEYS, how="left")

# --- Order columns clearly (Date text first for readability) ---
final = final[["All Dates (UTC)", "User Name", "User Email", dur_col]]

# --- Save & show ---
final.to_csv("combined_view_duration.csv", index=False)
print("✅ Final clean file saved as combined_view_duration.csv")
print(final.head(10).to_string(index=False))
print(final.dtypes)


Columns: ['Date(UTC)', 'User Name', 'User Email', 'View Duration (minutes)']
Shape: (174, 4)
            Date(UTC)          User Name             User Email View Duration (minutes)
Oct 28, 2025 05:56 PM        Ashton Dias      diasa2021@fau.edu                      37
Oct 29, 2025 12:50 AM Domenica Jaramillo djaramillo2023@fau.edu                      10
Oct 29, 2025 02:26 AM   Madison Peterkin  mpeterkin2020@fau.edu                      37
Oct 29, 2025 08:58 PM       Ethan Mclean    emclean2022@fau.edu                      37
Oct 30, 2025 12:22 AM     Steele Carlson   scarlson2018@fau.edu                      37
Oct 30, 2025 03:59 AM     Kelly Laguerre  klaguerre2014@fau.edu                     < 1
Oct 30, 2025 11:20 AM    Brianna Pilarte   bpilarte2020@fau.edu                      37
Oct 30, 2025 01:00 PM     Kelly Laguerre  klaguerre2014@fau.edu                     < 1
Oct 30, 2025 05:37 PM    Samantha Walter    walters2023@fau.edu                      14
Oct 30, 2025 09:38 PM      

C:\Users\pakal\AppData\Local\Temp\ipykernel_32620\3138696342.py:78: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  .agg(lambda s: ", ".join(sorted(pd.unique([x for x in s.dropna() if x]))))
